# Análisis del Balance de Importaciones

## Proyecto Subdirección de Logística

Este notebook documenta el proceso de exploración, análisis estadístico, control del proceso y modelado predictivo aplicado a información del balance de importaciones.

El flujo se desarrolla en Databricks sobre datos disponibles en un entorno Lakehouse, utilizando Apache Spark para la recuperación y persistencia de información y Python/Pandas para las etapas analíticas.


## Índice

### Actividades previas
- Instalación de librerías.
- Configuración del entorno.
- Importación de dependencias.
- Recuperación del conjunto de datos.

### 1. Análisis Exploratorio de Datos (EDA)
- Caracterización del DataFrame.
- Análisis de variables cuantitativas y cualitativas.
- Distribución de variables.
- Perfilamiento y visualización.
- Análisis de correlaciones.

### 2. Supuestos y análisis del proceso
- Análisis de causalidad.
- Gráficos de control Shewhart.
- Evaluación de capacidad y centramiento.

### 3. Modelado predictivo
- Prophet optimizado mediante Optuna.
- Pronóstico de `Vol_Nat_bls`.
- Pronóstico de `Vol_20_bls`.
- Pronóstico de `Temp_Prom`.
- Análisis de Z-Score.

> Los resultados generados por el notebook se almacenan nuevamente en tablas del entorno para su posterior consumo.


## Actividades previas al análisis

En esta sección se prepara el entorno de ejecución de Databricks, se instalan las dependencias necesarias y se recupera el conjunto de datos que será utilizado en las etapas posteriores.


In [ ]:
#Reiniciar el proceso de Python en un notebook, manteniendo el estado del cluster activo.
dbutils.library.restartPython()


In [ ]:
%pip install sweetviz # Perfilamiento de datos cualitativos
%pip install "ydata-profiling<4.6.0" # Perfilamiento de datos cuantitativos
%pip install klib # Limpieza, exploración y preprocesamiento de datos || pandas-profiling / polars
%pip install seaborn # Visualización estadística de los datos
%pip install --upgrade prophet cmdstanpy pystan
%pip install optuna
%pip install git+https://github.com/cmu-phil/causal-learn.git
%pip install pydot
%pip install graphviz
%pip install scikit-learn
%pip install ipython

In [ ]:
%sh apt-get update -y && apt-get install -y graphviz


In [ ]:
# Proyecto Analisis Importaciones

%matplotlib inline

import math
import pandas as pd
import numpy as np
import statistics
import itertools
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.pylab as plty
import plotly as py
import matplotlib.dates as mdates
from scipy.stats import zscore


#Librerias Necesarias de SPARK
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import *
from pyspark.sql.functions import udf, col, concat, concat_ws, explode, length, to_date, dayofyear, lower, regexp_replace, year, dayofmonth, lpad, lit, month, asc, desc, split
from pyspark.sql import functions as sf

from datetime import datetime

#Librerias de Calidad de Datos
#from ydata_profiling import ProfileReport
import sweetviz as sv
import klib

#Spark MLlib
from pyspark.ml import Pipeline
from pyspark.ml.feature import IndexToString, StringIndexer
from pyspark.ml.regression import AFTSurvivalRegression
from pyspark.ml.linalg import Vectors
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

#Librerias de Pronostico
import prophet
from prophet import Prophet
from prophet.diagnostics import performance_metrics

#EDA
from pandas_profiling import ProfileReport
import sweetviz as sv
import klib


In [ ]:
print(prophet.__version__)

In [ ]:
df = spark.table("direccion_negocio.direccion_general.v_siic_balance_importaciones")

display(df)


In [ ]:
df= df.toPandas()
df.head()

In [ ]:
df.dtypes

## Conversión y limpieza de datos

Antes de iniciar el análisis se estandarizan los tipos de datos, se convierten las fechas y variables cuantitativas y se eliminan columnas que no son necesarias para el análisis.

El procesamiento se realiza sobre una copia de trabajo para mantener separado el conjunto utilizado como fuente.


In [ ]:
#Conversion de tipo de Datos.

df['Clave_Centro'] = df['Clave_Centro'].astype(str)
df["Fecha"]=pd.to_datetime(df["Fecha"])
df['Vol_Nat_lts'] = pd.to_numeric(df['Vol_Nat_lts'], errors='coerce')
df['Vol_20_lts'] = pd.to_numeric(df['Vol_20_lts'], errors='coerce')
df['Vol_Nat_bls'] = pd.to_numeric(df['Vol_Nat_bls'], errors='coerce')
df['Vol_20_bls'] = pd.to_numeric(df['Vol_20_bls'], errors='coerce')
df['Temp_Prom'] = pd.to_numeric(df['Temp_Prom'], errors='coerce')


In [ ]:
#Validacion
df.dtypes

In [ ]:
#Ver los valores nulos.
print(df.isnull().sum())

In [ ]:
#Eliminar la Columna DateKey
if 'DateKey' in df.columns: df = df.drop('DateKey', axis=1)

# 1. Análisis Exploratorio de Datos (EDA)

Esta etapa busca comprender la estructura, distribución, calidad y relaciones entre las variables del balance de importaciones antes de aplicar técnicas estadísticas y modelos predictivos.


In [ ]:

from pandas_profiling import ProfileReport

EDA_Balance_Importaciones = ProfileReport(
    df,
    title='Balance de importaciones (EDA)',
    sensitive=True,
    duplicates=None,
    explorative=True,
    infer_dtypes=True,
    interactions=None,
    missing_diagrams=None,
    correlations={
        "auto": {"calculate": True},
        "pearson": {"calculate": True},
        "spearman": {"calculate": True}
    },
    minimal=True,
    orange_mode=True
)

displayHTML(EDA_Balance_Importaciones.to_html())

Sweetviz


In [ ]:
df1 = df.astype('category')
print(df1.dtypes)  # Verifica que sean category

In [ ]:
import sweetviz as sv# Generar análisis exploratorio con Sweetviz

reporte = sv.analyze(df1)

# Exportando el archivo en formato HTML
reporte.show_html("Reporte_Cualitativo_Importaciones.html")

# Mostrar el reporte
reporte.show_notebook()

In [ ]:
klib.missingval_plot(df) # returns a figure containing information about missing values

In [ ]:
# Grafico de la Distribucións del DataFrame
import klib
klib.dist_plot(df) 

### Caracterización del DataFrame

Se revisan las dimensiones, tipos de datos y principales características del conjunto de información antes de continuar con las etapas de modelado.


In [ ]:
df.dtypes

## Caracterización y análisis de variables

Se complementa la caracterización inicial mediante estadísticas descriptivas y visualizaciones orientadas a comprender el comportamiento de las variables del proceso.


In [ ]:
import seaborn as sns
sns.set_theme(style="ticks")

# Initialize the figure with a logarithmic x axis
f, ax = plt.subplots(figsize=(20, 45))
ax.set_xscale("log")

# Plot the orbital period with horizontal boxes
sns.boxplot(
    df, x="Vol_Nat_lts", y="Producto", hue="Medio_Transp",
    whis=[0, 100], width=.6
)

# Add in points to show each observation
sns.stripplot(df, x="Vol_Nat_lts", y="Producto", size=1, color="grey", jitter=0.1)

# Tweak the visual presentation
ax.yaxis.grid(True)
ax.set(ylabel="Producto")
sns.despine(trim=False, left=True)

In [ ]:
sns.set_theme(style="ticks")

# Initialize the figure with a logarithmic x axis
f, ax = plt.subplots(figsize=(20, 35))
ax.set_xscale("log")

# Plot the orbital period with horizontal boxes
sns.boxplot(
    df, x="Vol_20_lts", y="Producto", hue="Medio_Transp",
    whis=[0, 100], width=.6
)

# Add in points to show each observation
sns.stripplot(df, x="Vol_20_lts", y="Producto", size=1, hue="Medio_Transp", jitter=0.1)

# Tweak the visual presentation
ax.yaxis.grid(True)
ax.set(ylabel="Producto")
sns.despine(trim=False, left=True)

# 2. Supuestos y análisis del proceso

En esta etapa se aplican técnicas orientadas a explorar relaciones entre variables y evaluar el comportamiento estadístico del proceso.


### Análisis de causalidad

Se utiliza el PC Algorithm para explorar relaciones entre las variables numéricas disponibles. El análisis incluye limpieza, eliminación de variables constantes, reducción de correlación alta y estandarización antes de ejecutar el algoritmo.


In [ ]:
df.head()

In [ ]:

# ====================================
# 1. IMPORTS
# ====================================
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from sklearn.preprocessing import StandardScaler
import pandas as pd
from IPython.display import Image, display
import numpy as np
import pydot

# ====================================
# 2. PREPARAR DATASET
# ====================================
df_c = df.copy()

# Solo variables numéricas
df_c = df_c.select_dtypes(include=['float', 'int'])

# Eliminar columnas constantes
df_c = df_c.loc[:, df_c.std() > 0]

# Eliminar filas con NaN
df_c = df_c.dropna()

print("✅ Dimensiones iniciales:", df_c.shape)

# ====================================
# 3. ELIMINAR VARIABLES ALTAMENTE CORRELACIONADAS
# ====================================
corr = df_c.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df_c_clean = df_c.drop(columns=to_drop)

print("🧹 Columnas eliminadas por correlación > 0.95:")
print(to_drop)
print("✅ Dimensiones después de limpieza:", df_c_clean.shape)

# ====================================
# 4. ESCALAR DATOS
# ====================================
data_np = StandardScaler().fit_transform(df_c_clean.values)

# ====================================
# 5. EJECUTAR ALGORITMO PC
# ====================================
cg = pc(data_np, alpha=0.05)
graph = cg

# ====================================
# 6. GENERAR PNG DEL GRAFO
# ====================================
node_names = list(df_c_clean.columns)
pydot_graph = GraphUtils.to_pydot(graph.G, labels=node_names)
pydot_graph.write_png("causal_graph_display_labels.png")

# ====================================
# 7. MOSTRAR LA IMAGEN
# ====================================
display(Image(filename="causal_graph_display_labels.png"))

print("✅ Grafo causal generado correctamente: causal_graph_display_labels.png")

### Gráfico de control Shewhart — `Vol_Nat_bls`

Se analiza el comportamiento mensual de `Vol_Nat_bls` mediante promedios mensuales y límites de control de ±1σ, ±2σ y ±3σ.


In [ ]:
# --- Agrupación Mensual ---

# Agrupamos por Mes y calculamos el promedio de Vol_Nat_lts
df_schew_1 = (
    df.groupby(pd.Grouper(key="Fecha", freq="M"))["Vol_Nat_bls"]
    .mean()
    .reset_index()
)

# --- Cálculo de límites ---
data = df_schew_1["Vol_Nat_bls"]
media = statistics.mean(data)
desv_std = statistics.stdev(data)

# --- Crear DataFrame con límites por fila ---
df_shewhart = df_schew_1.copy()
df_shewhart["Media_Global"] = media
df_shewhart["+1σ"] = media + 1 * desv_std
df_shewhart["-1σ"] = media - 1 * desv_std
df_shewhart["+2σ"] = media + 2 * desv_std
df_shewhart["-2σ"] = media - 2 * desv_std
df_shewhart["+3σ"] = media + 3 * desv_std
df_shewhart["-3σ"] = media - 3 * desv_std

# --- Función para gráfico Shewhart con ±1σ, ±2σ, ±3σ ---
def grafico_shewhart(
    fechas, data,
    titulo="Gráfico de Control Shewhart Mensual (Vol_Nat_bls)"
):
    media = np.mean(data)
    std = np.std(data)
    plt.figure(figsize=(18, 10))

    # Serie de datos
    plt.plot(fechas, data, 'o-', color='blue', label='Promedio Mensual de Vol_Nat_bls')

    # Líneas de control
    plt.axhline(y=media, color='green', linestyle='-', linewidth=2, label='LC (Media)')
    plt.axhline(y=media + 1*std, color='magenta', linestyle='--', linewidth=1, label='+1σ')
    plt.axhline(y=media - 1*std, color='magenta', linestyle='--', linewidth=1, label='-1σ')
    plt.axhline(y=media + 2*std, color='brown', linestyle='-.', linewidth=1.2, label='+2σ')
    plt.axhline(y=media - 2*std, color='brown', linestyle='-.', linewidth=1.2, label='-2σ')
    plt.axhline(y=media + 3*std, color='red', linestyle=':', linewidth=1.5, label='+3σ (LCS)')
    plt.axhline(y=media - 3*std, color='red', linestyle=':', linewidth=1.5, label='-3σ (LCI)')

    # Configuración del gráfico
    plt.title(titulo, fontsize=18, weight='bold')
    plt.xlabel("Mes", fontsize=12)
    plt.ylabel("Vol_Nat_bls Promedio", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

# --- Llamada al gráfico ---
grafico_shewhart(df_schew_1["Fecha"], df_schew_1["Vol_Nat_bls"])

# --- Mostrar y exportar resultados ---
print("\n✅ DataFrame con resultados Shewhart:\n")
print(df_shewhart.head())


In [ ]:
df_shewhart.head()

In [ ]:
df_shewhart.tail()

In [ ]:
# Persistencia de resultados del control estadístico en el Lakehouse
# Convert Pandas DataFrame to Spark DataFrame
df_spark_shewhart = spark.createDataFrame(df_shewhart)

# Save the Spark DataFrame as a table in the specified schema
df_spark_shewhart.write.mode("overwrite").saveAsTable("direccion_negocio.direccion_general.Shewhart_balance_importaciones_Vol_Nat_bls")


### Evaluación de control y capacidad del proceso — `Vol_Nat_bls`

Se calculan indicadores de capacidad (`Cp`, `Cpk`), centramiento (`K`) y una prueba de normalidad para generar una evaluación general del proceso.


In [ ]:
# Si no tienes scipy instalado, descomenta la siguiente línea:
# %pip install scipy

from scipy import stats
import numpy as np

# --- Datos base ---
datos = df["Vol_Nat_bls"]
LSE = df_shewhart["+3σ"].iloc[-1]
LIE = df_shewhart["-3σ"].iloc[-1]

# --- Función principal ---
def evaluar_estado_proceso(
    datos,
    LSE,
    LIE,
    alpha=0.05
):
    # Análisis de capacidad
    media = np.mean(datos)
    std = np.std(datos, ddof=1)
    objetivo = (LSE + LIE) / 2
    Cp = (LSE - LIE) / (6 * std)
    Cpk = min((LSE - media) / (3 * std), (media - LIE) / (3 * std))
    K = abs((media - objetivo) / ((LSE - LIE) / 2)) * 100

    # Test de normalidad
    _, p_normal = stats.normaltest(datos)

    # Evaluación
    evaluacion = {
        'CAPAZ': Cpk >= 1.33,
        'ACEPTABLE': Cpk >= 1.0,
        'BIEN_CENTRADO': K <= 20,
        'NORMAL': p_normal > alpha,
        'ESTABLE': abs(Cp - Cpk) < 0.3
    }

    # Estado general
    criterios_cumplidos = sum(evaluacion.values())
    estado_general = "CONTROLADO" if criterios_cumplidos >= 4 else "NO CONTROLADO"

    return {
        'metricas': {'Cp': Cp, 'Cpk': Cpk, 'K': K, 'Media': media, 'Std': std},
        'evaluacion': evaluacion,
        'estado': estado_general,
        'recomendaciones': generar_recomendaciones(evaluacion, Cpk, K)
    }

# --- Generar recomendaciones en lista ---
def generar_recomendaciones(
    evaluacion,
    Cpk,
    K
):
    recomendaciones = []
    if not evaluacion['CAPAZ']:
        if Cpk < 1.0:
            recomendaciones.extend([
                "Reducir variabilidad del proceso",
                "Inspección 100% del producto",
                "Parada del proceso para análisis",
                "Ajuste/recalibración completa",
                "Asignación de equipo de mejora",
                "Revisión de especificaciones con cliente"
            ])
        elif Cpk < 1.33:
            recomendaciones.extend([
              "Mejorar capacidad del proceso",
              "Implementar SPC (Control Estadístico de Proceso)",
              "Revisar y ajustar parámetros críticos",
              "Capacitación operativa reforzada",
              "Aumentar frecuencia de muestreo",
              "Análisis de capacidad por característica",
              "Mejora de herramientas y equipos",
              "Estandarización de métodos"])
    if not evaluacion['BIEN_CENTRADO']:
        recomendaciones.append("Ajustar centramiento del proceso")
    if not evaluacion['NORMAL']:
        recomendaciones.append("Investigar causas de no-normalidad")
    return recomendaciones

# --- Uso completo ---
resultado = evaluar_estado_proceso(datos, LSE, LIE)

# --- Mostrar resultados ---
print("=== RESUMEN DEL PROCESO ===")
print(f"Estado general del proceso: {resultado['estado']}")
print(f"Cp: {resultado['metricas']['Cp']:.3f}")
print(f"Cpk: {resultado['metricas']['Cpk']:.3f}")
print(f"Desviación estándar: {resultado['metricas']['Std']:.3f}")
print(f"Centramiento (K): {resultado['metricas']['K']:.1f}%")

print("\n📋 Recomendaciones:")
for i, rec in enumerate(resultado['recomendaciones'], 1):
    print(f"{i}. {rec}")


### Gráfico de control Shewhart — `Vol_20_bls`

Se analiza el comportamiento mensual de `Vol_20_bls` mediante promedios mensuales y límites de control de ±1σ, ±2σ y ±3σ.


In [ ]:

# --- Agrupación Mensual ---

# Agrupamos por Mes y calculamos el promedio de Vol_20_bls
df_schew_2 = (
    df.groupby(pd.Grouper(key="Fecha", freq="M"))["Vol_20_bls"]
    .mean()
    .reset_index()
)

# --- Cálculo de límites ---
data = df_schew_2["Vol_20_bls"]
media = statistics.mean(data)
desv_std = statistics.stdev(data)

# --- Crear DataFrame con límites por fila ---
df_shewhart = df_schew_2.copy()
df_shewhart["Media_Global"] = media
df_shewhart["+1σ"] = media + 1 * desv_std
df_shewhart["-1σ"] = media - 1 * desv_std
df_shewhart["+2σ"] = media + 2 * desv_std
df_shewhart["-2σ"] = media - 2 * desv_std
df_shewhart["+3σ"] = media + 3 * desv_std
df_shewhart["-3σ"] = media - 3 * desv_std

# --- Función para gráfico Shewhart con ±1σ, ±2σ, ±3σ ---
def grafico_shewhart(
    fechas, data,
    titulo="Gráfico de Control Shewhart Mensual (Vol_20_bls)"
):
    media = np.mean(data)
    std = np.std(data)
    plt.figure(figsize=(18, 10))

    # Serie de datos
    plt.plot(fechas, data, 'o-', color='blue', label='Promedio Mensual de Vol_20_bls')

    # Líneas de control
    plt.axhline(y=media, color='green', linestyle='-', linewidth=2, label='LC (Media)')
    plt.axhline(y=media + 1*std, color='magenta', linestyle='--', linewidth=1, label='+1σ')
    plt.axhline(y=media - 1*std, color='magenta', linestyle='--', linewidth=1, label='-1σ')
    plt.axhline(y=media + 2*std, color='brown', linestyle='-.', linewidth=1.2, label='+2σ')
    plt.axhline(y=media - 2*std, color='brown', linestyle='-.', linewidth=1.2, label='-2σ')
    plt.axhline(y=media + 3*std, color='red', linestyle=':', linewidth=1.5, label='+3σ (LCS)')
    plt.axhline(y=media - 3*std, color='red', linestyle=':', linewidth=1.5, label='-3σ (LCI)')

    # Configuración del gráfico
    plt.title(titulo, fontsize=18, weight='bold')
    plt.xlabel("Mes", fontsize=12)
    plt.ylabel("Vol_20_bls Promedio", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

# --- Llamada al gráfico ---
grafico_shewhart(df_schew_2["Fecha"], df_schew_2["Vol_20_bls"])

# --- Mostrar y exportar resultados ---
print("\n✅ DataFrame con resultados Shewhart:\n")
print(df_shewhart.head())


In [ ]:
df_shewhart.head()

In [ ]:
df_shewhart.tail()

In [ ]:
# Persistencia de resultados del control estadístico en el Lakehouse
# Convert Pandas DataFrame to Spark DataFrame
df_spark_shewhart = spark.createDataFrame(df_shewhart)

# Save the Spark DataFrame as a table in the specified schema
df_spark_shewhart.write.mode("overwrite").saveAsTable("direccion_negocio.direccion_general.Shewhart_balance_importaciones_Vol_20_bls")


### Evaluación de control y capacidad del proceso — `Vol_20_bls`

Se aplican los mismos indicadores de capacidad, centramiento y normalidad sobre `Vol_20_bls`.


In [ ]:
# Si no tienes scipy instalado, descomenta la siguiente línea:
# %pip install scipy

from scipy import stats
import numpy as np

# --- Datos base ---
datos = df["Vol_20_bls"]
LSE = df_shewhart["+3σ"].iloc[-1]
LIE = df_shewhart["-3σ"].iloc[-1]

# --- Función principal ---
def evaluar_estado_proceso(
    datos,
    LSE,
    LIE,
    alpha=0.05
):
    # Análisis de capacidad
    media = np.mean(datos)
    std = np.std(datos, ddof=1)
    objetivo = (LSE + LIE) / 2
    Cp = (LSE - LIE) / (6 * std)
    Cpk = min((LSE - media) / (3 * std), (media - LIE) / (3 * std))
    K = abs((media - objetivo) / ((LSE - LIE) / 2)) * 100

    # Test de normalidad
    _, p_normal = stats.normaltest(datos)

    # Evaluación
    evaluacion = {
        'CAPAZ': Cpk >= 1.33,
        'ACEPTABLE': Cpk >= 1.0,
        'BIEN_CENTRADO': K <= 20,
        'NORMAL': p_normal > alpha,
        'ESTABLE': abs(Cp - Cpk) < 0.3
    }

    # Estado general
    criterios_cumplidos = sum(evaluacion.values())
    estado_general = "CONTROLADO" if criterios_cumplidos >= 4 else "NO CONTROLADO"

    return {
        'metricas': {'Cp': Cp, 'Cpk': Cpk, 'K': K, 'Media': media, 'Std': std},
        'evaluacion': evaluacion,
        'estado': estado_general,
        'recomendaciones': generar_recomendaciones(evaluacion, Cpk, K)
    }

# --- Generar recomendaciones en lista ---
def generar_recomendaciones(
    evaluacion,
    Cpk,
    K
):
    recomendaciones = []
    if not evaluacion['CAPAZ']:
        if Cpk < 1.0:
            recomendaciones.extend([
                "Reducir variabilidad del proceso",
                "Inspección 100% del producto",
                "Parada del proceso para análisis",
                "Ajuste/recalibración completa",
                "Asignación de equipo de mejora",
                "Revisión de especificaciones con cliente"
            ])
        elif Cpk < 1.33:
            recomendaciones.extend([
              "Mejorar capacidad del proceso",
              "Implementar SPC (Control Estadístico de Proceso)",
              "Revisar y ajustar parámetros críticos",
              "Capacitación operativa reforzada",
              "Aumentar frecuencia de muestreo",
              "Análisis de capacidad por característica",
              "Mejora de herramientas y equipos",
              "Estandarización de métodos"])
    if not evaluacion['BIEN_CENTRADO']:
        recomendaciones.append("Ajustar centramiento del proceso")
    if not evaluacion['NORMAL']:
        recomendaciones.append("Investigar causas de no-normalidad")
    return recomendaciones

# --- Uso completo ---
resultado = evaluar_estado_proceso(datos, LSE, LIE)

# --- Mostrar resultados ---
print("=== RESUMEN DEL PROCESO ===")
print(f"Estado general del proceso: {resultado['estado']}")
print(f"Cp: {resultado['metricas']['Cp']:.3f}")
print(f"Cpk: {resultado['metricas']['Cpk']:.3f}")
print(f"Desviación estándar: {resultado['metricas']['Std']:.3f}")
print(f"Centramiento (K): {resultado['metricas']['K']:.1f}%")

print("\n📋 Recomendaciones:")
for i, rec in enumerate(resultado['recomendaciones'], 1):
    print(f"{i}. {rec}")


### Gráfico de control Shewhart — `Temp_Prom`

Se analiza el comportamiento semanal de `Temp_Prom` mediante promedios semanales y límites de control de ±1σ, ±2σ y ±3σ.


In [ ]:
# --- Agrupación Semanal ---

# Agrupamos por Mes y calculamos el promedio de Temp_Prom
df_schew_3 = (
    df.groupby(pd.Grouper(key="Fecha", freq="W"))["Temp_Prom"]
    .mean()
    .reset_index()
)

# --- Cálculo de límites ---
data = df_schew_3["Temp_Prom"]
media = statistics.mean(data)
desv_std = statistics.stdev(data)

# --- Crear DataFrame con límites por fila ---
df_shewhart = df_schew_3.copy()
df_shewhart["Media_Global"] = media
df_shewhart["+1σ"] = media + 1 * desv_std
df_shewhart["-1σ"] = media - 1 * desv_std
df_shewhart["+2σ"] = media + 2 * desv_std
df_shewhart["-2σ"] = media - 2 * desv_std
df_shewhart["+3σ"] = media + 3 * desv_std
df_shewhart["-3σ"] = media - 3 * desv_std

# --- Función para gráfico Shewhart con ±1σ, ±2σ, ±3σ ---
def grafico_shewhart(
    fechas, data,
    titulo="Gráfico de Control Shewhart Semanal (Temp_Prom)"
):
    media = np.mean(data)
    std = np.std(data)
    plt.figure(figsize=(18, 10))

    # Serie de datos
    plt.plot(fechas, data, 'o-', color='blue', label='Promedio Semanal de Temp_Prom')

    # Líneas de control
    plt.axhline(y=media, color='green', linestyle='-', linewidth=2, label='LC (Media)')
    plt.axhline(y=media + 1*std, color='magenta', linestyle='--', linewidth=1, label='+1σ')
    plt.axhline(y=media - 1*std, color='magenta', linestyle='--', linewidth=1, label='-1σ')
    plt.axhline(y=media + 2*std, color='brown', linestyle='-.', linewidth=1.2, label='+2σ')
    plt.axhline(y=media - 2*std, color='brown', linestyle='-.', linewidth=1.2, label='-2σ')
    plt.axhline(y=media + 3*std, color='red', linestyle=':', linewidth=1.5, label='+3σ (LCS)')
    plt.axhline(y=media - 3*std, color='red', linestyle=':', linewidth=1.5, label='-3σ (LCI)')

    # Configuración del gráfico
    plt.title(titulo, fontsize=22, weight='bold')
    plt.xlabel("Semana", fontsize=12)
    plt.ylabel("Temp_Prom Promedio", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

# --- Llamada al gráfico ---
grafico_shewhart(df_schew_3["Fecha"], df_schew_3["Temp_Prom"])

# --- Mostrar y exportar resultados ---
print("\n✅ DataFrame con resultados Shewhart:\n")
print(df_shewhart.head())


In [ ]:
df_shewhart.head()

In [ ]:
df_shewhart.tail()

In [ ]:
# Persistencia de resultados del control estadístico en el Lakehouse
# Convert Pandas DataFrame to Spark DataFrame
df_spark_shewhart = spark.createDataFrame(df_shewhart)

# Save the Spark DataFrame as a table in the specified schema
df_spark_shewhart.write.mode("overwrite").saveAsTable("direccion_negocio.direccion_general.Shewhart_balance_importaciones_Temp_Prom")


### Evaluación de control y capacidad del proceso — `Temp_Prom`

Se evalúan capacidad, centramiento y normalidad para la variable `Temp_Prom`.


In [ ]:
# Si no tienes scipy instalado, descomenta la siguiente línea:
# %pip install scipy

from scipy import stats
import numpy as np

# --- Datos base ---
datos = df["Temp_Prom"]
LSE = df_shewhart["+3σ"].iloc[-1]
LIE = df_shewhart["-3σ"].iloc[-1]

# --- Función principal ---
def evaluar_estado_proceso(
    datos,
    LSE,
    LIE,
    alpha=0.05
):
    # Análisis de capacidad
    media = np.mean(datos)
    std = np.std(datos, ddof=1)
    objetivo = (LSE + LIE) / 2
    Cp = (LSE - LIE) / (6 * std)
    Cpk = min((LSE - media) / (3 * std), (media - LIE) / (3 * std))
    K = abs((media - objetivo) / ((LSE - LIE) / 2)) * 100

    # Test de normalidad
    _, p_normal = stats.normaltest(datos)

    # Evaluación
    evaluacion = {
        'CAPAZ': Cpk >= 1.33,
        'ACEPTABLE': Cpk >= 1.0,
        'BIEN_CENTRADO': K <= 20,
        'NORMAL': p_normal > alpha,
        'ESTABLE': abs(Cp - Cpk) < 0.3
    }

    # Estado general
    criterios_cumplidos = sum(evaluacion.values())
    estado_general = "CONTROLADO" if criterios_cumplidos >= 4 else "NO CONTROLADO"

    return {
        'metricas': {'Cp': Cp, 'Cpk': Cpk, 'K': K, 'Media': media, 'Std': std},
        'evaluacion': evaluacion,
        'estado': estado_general,
        'recomendaciones': generar_recomendaciones(evaluacion, Cpk, K)
    }

# --- Generar recomendaciones en lista ---
def generar_recomendaciones(
    evaluacion,
    Cpk,
    K
):
    recomendaciones = []
    if not evaluacion['CAPAZ']:
        if Cpk < 1.0:
            recomendaciones.extend([
                "Reducir variabilidad del proceso",
                "Inspección 100% del producto",
                "Parada del proceso para análisis",
                "Ajuste/recalibración completa",
                "Asignación de equipo de mejora",
                "Revisión de especificaciones con cliente"
            ])
        elif Cpk < 1.33:
            recomendaciones.extend([
              "Mejorar capacidad del proceso",
              "Implementar SPC (Control Estadístico de Proceso)",
              "Revisar y ajustar parámetros críticos",
              "Capacitación operativa reforzada",
              "Aumentar frecuencia de muestreo",
              "Análisis de capacidad por característica",
              "Mejora de herramientas y equipos",
              "Estandarización de métodos"])
    if not evaluacion['BIEN_CENTRADO']:
        recomendaciones.append("Ajustar centramiento del proceso")
    if not evaluacion['NORMAL']:
        recomendaciones.append("Investigar causas de no-normalidad")
    return recomendaciones

# --- Uso completo ---
resultado = evaluar_estado_proceso(datos, LSE, LIE)

# --- Mostrar resultados ---
print("=== RESUMEN DEL PROCESO ===")
print(f"Estado general del proceso: {resultado['estado']}")
print(f"Cp: {resultado['metricas']['Cp']:.3f}")
print(f"Cpk: {resultado['metricas']['Cpk']:.3f}")
print(f"Desviación estándar: {resultado['metricas']['Std']:.3f}")
print(f"Centramiento (K): {resultado['metricas']['K']:.1f}%")

print("\n📋 Recomendaciones:")
for i, rec in enumerate(resultado['recomendaciones'], 1):
    print(f"{i}. {rec}")


# 3. Modelos predictivos

Se desarrollan pronósticos de 30 días mediante **Prophet**. La selección de hiperparámetros se realiza con **Optuna**, utilizando MAPE como función objetivo sobre las observaciones utilizadas para ajustar el modelo.

> **Consideración metodológica:** el MAPE utilizado durante la optimización se calcula sobre valores históricos incluidos en el entrenamiento, por lo que debe interpretarse como una métrica *in-sample* y no como una evaluación de generalización sobre un conjunto de prueba independiente.


### Modelo Prophet + Optuna — `Vol_Nat_bls`

Se prepara la serie temporal, se optimizan hiperparámetros de Prophet mediante 40 iteraciones de Optuna y posteriormente se entrena el modelo final para generar un horizonte de 30 días.


In [ ]:
import optuna
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================
# PREPARAR DATA
# ==============================
df_opt = df.copy()
df_opt['Fecha'] = pd.to_datetime(df_opt['Fecha'])
df_opt = df_opt[['Fecha', 'Vol_Nat_bls']].dropna()

df_opt  = df_opt .rename(columns={'Fecha':'ds', 'Vol_Nat_bls':'y'})
df_opt  = df_opt [df_opt ['y'] > 0]  # evitar divisiones y log en cero

# ==============================
# FUNCIÓN OBJETIVO ESTABLE
# ==============================
def objective(trial):

    params = {
        "growth": "linear",
        "changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.001, 0.2),
        "seasonality_prior_scale": trial.suggest_float("seasonality_prior_scale", 0.01, 5.0),
        "holidays_prior_scale": trial.suggest_float("holidays_prior_scale", 0.01, 5.0),
        "seasonality_mode": trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]),
        "changepoint_range": trial.suggest_float("changepoint_range", 0.6, 0.9),
        "yearly_seasonality": True,
        "weekly_seasonality": True,
        "daily_seasonality": True
    }

    try:
        model = Prophet(**params)
        model.fit(df_opt)

        future = model.make_future_dataframe(periods=30, freq="D")
        forecast = model.predict(future)

        merged = pd.merge(df_opt, forecast[['ds','yhat']], on='ds', how='inner')
        mape = mean_absolute_percentage_error(merged['y'], merged['yhat'])

        if np.isnan(mape) or np.isinf(mape) or mape > 10:  # fuera de rango razonable
            return 10

        return mape

    except Exception:
        return 10   # castigo en Optuna para modelos inestables

# ==============================
# Optimización de hiperparámetros
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("✅ BEST PARAMS:")
print(study.best_params)
print("✅ BEST MAPE:", study.best_value)

# ==============================
# ENTRENAR MODELO FINAL
# ==============================
best_params = study.best_params
model = Prophet(**best_params)
model.fit(df_opt)

future = model.make_future_dataframe(periods=30, freq="D")
forecast = model.predict(future)

fig1 = model.plot(forecast)
fig2 = model.plot_components(forecast)


In [ ]:
# ================================
# Merge
# ================================
# Merge directo con fechas
df_merge1 = pd.merge(
    df,
    forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'trend', 'trend_lower', 'trend_upper', 'additive_terms', 'additive_terms_lower', 'additive_terms_upper', 'multiplicative_terms', 'multiplicative_terms_lower', 'multiplicative_terms_upper' , 'weekly', 'weekly_lower', 'weekly_upper', 'yearly', 'yearly_lower', 'yearly_upper']],
    left_on="Fecha",  # columna en df_original
    right_on="ds",    # columna en forecast
    how="left"        # left: conserva todas las fechas de df_original
)

# ================================
# Limpiar columna duplicada de fechas
# ================================
# 1. Eliminar las columnas ds_x, ds_y, y
df_merge1.drop(columns=['ds_x', 'ds_y', 'y', 'ds'], inplace=True, errors='ignore')


print(df_merge1)

In [ ]:
df_merge1

### Análisis de Z-Score — `Vol_Nat_bls`

Se calcula el Z-Score de `yhat` para identificar observaciones del pronóstico que se encuentran alejadas de la media en términos de desviaciones estándar.


In [ ]:
# Análisis de Z-SCORE - Vol_Nat_bls

from scipy.stats import zscore


# Calcular z-score para la columna de pronostico
df_merge1["zscore_Vol_Nat_bls"] = zscore(df_merge1["yhat"])

In [ ]:
# Persistencia del conjunto final con pronóstico y Z-Score en el Lakehouse
# Convert Pandas DataFrame to Spark DataFrame
df_spark = spark.createDataFrame(df_merge1)

# Save the Spark DataFrame as a table in the specified schema
df_spark.write.mode("overwrite").saveAsTable("direccion_negocio.direccion_general.Zscore_balance_importaciones_Vol_Nat_bls")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Asegúrate de que la columna Fecha sea datetime
df_merge1['Fecha'] = pd.to_datetime(df_merge1['Fecha'])

# Crear columna para indicar si está dentro o fuera del rango de 3σ
df_merge1['estado'] = df_merge1['zscore_Vol_Nat_bls'].apply(lambda x: 'Fuera' if abs(x) > 3 else 'Dentro')

# Graficar
plt.figure(figsize=(25,20))
sns.scatterplot(
    data=df_merge1, 
    x='Fecha', 
    y='zscore_Vol_Nat_bls', 
    hue='estado',
    style='Centro', 
    palette={'Dentro':'blue','Fuera':'red'}, 
    s=100
)

# === Líneas de control ===
# 1σ
plt.axhline(1, color='green', linestyle='--', label='+1σ')
plt.axhline(-1, color='green', linestyle='--', label='-1σ')

# 2σ
plt.axhline(2, color='orange', linestyle='--', label='+2σ')
plt.axhline(-2, color='orange', linestyle='--', label='-2σ')

# 3σ
plt.axhline(3, color='red', linestyle='--', label='+3σ')
plt.axhline(-3, color='red', linestyle='--', label='-3σ')

# === Personalización ===
plt.title("Z-Score de Vol_Nat_bls por Centro", size=30)
plt.ylabel("Z-Score", size=20)
plt.xlabel("Fecha", size=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)  # Mueve la leyenda a la derecha
plt.xticks(rotation=45)  # Rotar fechas para mejor visualización
plt.tight_layout()
plt.show()


### Modelo Prophet + Optuna — `Vol_20_bls`

Se repite el flujo de optimización y pronóstico para la variable `Vol_20_bls`, manteniendo un horizonte de 30 días.


In [ ]:
import optuna
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================
# PREPARAR DATA
# ==============================
df_opt1 = df.copy()
df_opt1['Fecha'] = pd.to_datetime(df_opt1['Fecha'])
df_opt1 = df_opt1[['Fecha', 'Vol_20_bls']].dropna()

df_opt1  = df_opt1 .rename(columns={'Fecha':'ds', 'Vol_20_bls':'y'})
df_opt1  = df_opt1 [df_opt1 ['y'] > 0]  # evitar divisiones y log en cero

# ==============================
# FUNCIÓN OBJETIVO ESTABLE
# ==============================
def objective(trial):

    params = {
        "growth": "linear",
        "changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.001, 0.2),
        "seasonality_prior_scale": trial.suggest_float("seasonality_prior_scale", 0.01, 5.0),
        "holidays_prior_scale": trial.suggest_float("holidays_prior_scale", 0.01, 5.0),
        "seasonality_mode": trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]),
        "changepoint_range": trial.suggest_float("changepoint_range", 0.6, 0.9),
        "yearly_seasonality": True,
        "weekly_seasonality": True,
        "daily_seasonality": True
    }

    try:
        model = Prophet(**params)
        model.fit(df_opt1)

        future = model.make_future_dataframe(periods=30, freq="D")
        forecast = model.predict(future)

        merged = pd.merge(df_opt1, forecast[['ds','yhat']], on='ds', how='inner')
        mape = mean_absolute_percentage_error(merged['y'], merged['yhat'])

        if np.isnan(mape) or np.isinf(mape) or mape > 10:  # fuera de rango razonable
            return 10

        return mape

    except Exception:
        return 10   # castigo en Optuna para modelos inestables

# ==============================
# Optimización de hiperparámetros
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("✅ BEST PARAMS:")
print(study.best_params)
print("✅ BEST MAPE:", study.best_value)

# ==============================
# ENTRENAR MODELO FINAL
# ==============================
best_params = study.best_params
model = Prophet(**best_params)
model.fit(df_opt1)

future = model.make_future_dataframe(periods=30, freq="D")
forecast = model.predict(future)

fig1 = model.plot(forecast)
fig2 = model.plot_components(forecast)


In [ ]:
# ================================
# Merge
# ================================
# Merge directo con fechas
df_merge2 = pd.merge(
    df,
    forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'trend', 'trend_lower', 'trend_upper', 'additive_terms', 'additive_terms_lower', 'additive_terms_upper', 'multiplicative_terms', 'multiplicative_terms_lower', 'multiplicative_terms_upper' , 'weekly', 'weekly_lower', 'weekly_upper', 'yearly', 'yearly_lower', 'yearly_upper']],
    left_on="Fecha",  # columna en df_original
    right_on="ds",    # columna en forecast
    how="left"        # left: conserva todas las fechas de df_original
)

# ================================
# Limpiar columna duplicada de fechas
# ================================
# 1. Eliminar las columnas ds_x, ds_y, y
df_merge2.drop(columns=['ds_x', 'ds_y', 'y', 'ds'], inplace=True, errors='ignore')


print(df_merge2)

### Análisis de Z-Score — `Vol_20_bls`

Se calcula el Z-Score de los valores pronosticados por Prophet.


In [ ]:
# Análisis de Z-SCORE - Vol_20_bls

from scipy.stats import zscore


# Calcular z-score para la columna de pronostico
df_merge2["zscore_Vol_20_bls"] = zscore(df_merge2["yhat"])

In [ ]:
# Persistencia del conjunto final con pronóstico y Z-Score en el Lakehouse
# Convert Pandas DataFrame to Spark DataFrame
df_spark = spark.createDataFrame(df_merge2)

# Save the Spark DataFrame as a table in the specified schema
df_spark.write.mode("overwrite").saveAsTable("direccion_negocio.direccion_general.Zscore_balance_importaciones_Vol_20_bls")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Asegúrate de que la columna Fecha sea datetime
df_merge2['Fecha'] = pd.to_datetime(df_merge2['Fecha'])

# Crear columna para indicar si está dentro o fuera del rango de 3σ
df_merge2['estado'] = df_merge2['zscore_Vol_20_bls'].apply(lambda x: 'Fuera' if abs(x) > 3 else 'Dentro')

# Graficar
plt.figure(figsize=(25,20))
sns.scatterplot(
    data=df_merge2, 
    x='Fecha', 
    y='zscore_Vol_20_bls', 
    hue='estado',
    style='Centro', 
    palette={'Dentro':'blue','Fuera':'red'}, 
    s=100
)

# === Líneas de control ===
# 1σ
plt.axhline(1, color='green', linestyle='--', label='+1σ')
plt.axhline(-1, color='green', linestyle='--', label='-1σ')

# 2σ
plt.axhline(2, color='orange', linestyle='--', label='+2σ')
plt.axhline(-2, color='orange', linestyle='--', label='-2σ')

# 3σ
plt.axhline(3, color='red', linestyle='--', label='+3σ')
plt.axhline(-3, color='red', linestyle='--', label='-3σ')

# === Personalización ===
plt.title("Z-Score de Vol_20_bls por Centro", size=30)
plt.ylabel("Z-Score", size=20)
plt.xlabel("Fecha", size=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)  # Mueve la leyenda a la derecha
plt.xticks(rotation=45)  # Rotar fechas para mejor visualización
plt.tight_layout()
plt.show()


### Modelo Prophet + Optuna — `Temp_Prom`

Se repite el flujo de optimización y pronóstico para `Temp_Prom`, manteniendo un horizonte de 30 días.


In [ ]:
import optuna
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ==============================
# PREPARAR DATA
# ==============================
df_opt2 = df.copy()
df_opt2['Fecha'] = pd.to_datetime(df_opt2['Fecha'])
df_opt2 = df_opt2[['Fecha', 'Temp_Prom']].dropna()

df_opt2  = df_opt2 .rename(columns={'Fecha':'ds', 'Temp_Prom':'y'})
df_opt2  = df_opt2 [df_opt2 ['y'] > 0]  # evitar divisiones y log en cero

# ==============================
# FUNCIÓN OBJETIVO ESTABLE
# ==============================
def objective(trial):

    params = {
        "growth": "linear",
        "changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.001, 0.2),
        "seasonality_prior_scale": trial.suggest_float("seasonality_prior_scale", 0.01, 5.0),
        "holidays_prior_scale": trial.suggest_float("holidays_prior_scale", 0.01, 5.0),
        "seasonality_mode": trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]),
        "changepoint_range": trial.suggest_float("changepoint_range", 0.6, 0.9),
        "yearly_seasonality": True,
        "weekly_seasonality": True,
        "daily_seasonality": True
    }

    try:
        model = Prophet(**params)
        model.fit(df_opt2)

        future = model.make_future_dataframe(periods=30, freq="D")
        forecast = model.predict(future)

        merged = pd.merge(df_opt2, forecast[['ds','yhat']], on='ds', how='inner')
        mape = mean_absolute_percentage_error(merged['y'], merged['yhat'])

        if np.isnan(mape) or np.isinf(mape) or mape > 10:  # fuera de rango razonable
            return 10

        return mape

    except Exception:
        return 10   # castigo en Optuna para modelos inestables

# ==============================
# Optimización de hiperparámetros
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("✅ BEST PARAMS:")
print(study.best_params)
print("✅ BEST MAPE:", study.best_value)

# ==============================
# ENTRENAR MODELO FINAL
# ==============================
best_params = study.best_params
model = Prophet(**best_params)
model.fit(df_opt2)

future = model.make_future_dataframe(periods=30, freq="D")
forecast = model.predict(future)

fig1 = model.plot(forecast)
fig2 = model.plot_components(forecast)


In [ ]:
# ================================
# Merge
# ================================
# Merge directo con fechas
df_merge3 = pd.merge(
    df,
    forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'trend', 'trend_lower', 'trend_upper', 'additive_terms', 'additive_terms_lower', 'additive_terms_upper', 'multiplicative_terms', 'multiplicative_terms_lower', 'multiplicative_terms_upper' , 'weekly', 'weekly_lower', 'weekly_upper', 'yearly', 'yearly_lower', 'yearly_upper']],
    left_on="Fecha",  # columna en df_original
    right_on="ds",    # columna en forecast
    how="left"        # left: conserva todas las fechas de df_original
)

# ================================
# Limpiar columna duplicada de fechas
# ================================
# 1. Eliminar las columnas ds_x, ds_y, y
df_merge3.drop(columns=['ds_x', 'ds_y', 'y', 'ds'], inplace=True, errors='ignore')


print(df_merge3)

### Análisis de Z-Score — `Temp_Prom`

Se calcula el Z-Score de los valores pronosticados de `Temp_Prom` para facilitar la identificación de valores alejados de la distribución central.


In [ ]:
## Analisis del Z - SCORE Temp_Prom

from scipy.stats import zscore


# Calcular z-score para la columna de pronostico
df_merge3["zscore_Temp_Prom"] = zscore(df_merge3["yhat"])

In [ ]:
# Persistencia del conjunto final con pronóstico y Z-Score en el Lakehouse
# Convert Pandas DataFrame to Spark DataFrame
df_spark = spark.createDataFrame(df_merge3)

# Save the Spark DataFrame as a table in the specified schema
df_spark.write.mode("overwrite").saveAsTable("direccion_negocio.direccion_general.Zscore_balance_importaciones_Temp_Prom")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Asegúrate de que la columna Fecha sea datetime
df_merge3['Fecha'] = pd.to_datetime(df_merge3['Fecha'])

# Crear columna para indicar si está dentro o fuera del rango de 3σ
df_merge3['estado'] = df_merge3['zscore_Temp_Prom'].apply(lambda x: 'Fuera' if abs(x) > 3 else 'Dentro')

# Graficar
plt.figure(figsize=(25,20))
sns.scatterplot(
    data=df_merge3, 
    x='Fecha', 
    y='zscore_Temp_Prom', 
    hue='estado',
    style='Centro', 
    palette={'Dentro':'blue','Fuera':'red'}, 
    s=100
)

# === Líneas de control ===
# 1σ
plt.axhline(1, color='green', linestyle='--', label='+1σ')
plt.axhline(-1, color='green', linestyle='--', label='-1σ')

# 2σ
plt.axhline(2, color='orange', linestyle='--', label='+2σ')
plt.axhline(-2, color='orange', linestyle='--', label='-2σ')

# 3σ
plt.axhline(3, color='red', linestyle='--', label='+3σ')
plt.axhline(-3, color='red', linestyle='--', label='-3σ')

# === Personalización ===
plt.title("Z-Score de Temp_Prom por Centro", size=30)
plt.ylabel("Z-Score", size=20)
plt.xlabel("Fecha", size=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)  # Mueve la leyenda a la derecha
plt.xticks(rotation=45)  # Rotar fechas para mejor visualización
plt.tight_layout()
plt.show()
